> Find clusters of windows using graphs (*communities*)

In [1]:
import pathlib
import random

import sklearn.neighbors
import igraph as ig
import leidenalg

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from bokeh.palettes import Category20
from bokeh.plotting import show
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from bokeh.models import Label

from IPython.display import display, Markdown

import trapallada.plot
import trapallada.analysis

In [ ]:
output_notebook(resources=INLINE)
# output_notebook()

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [4]:
input_files_stem = '../results/epochs/20000/4_sources/epochs'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs_05_x_sd'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs' # rotated "C" below

It is turned into a `Pathlib`

In [ ]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

Files to *cache* the construction of the graph

In [7]:
# nodes_file = pathlib.Path('nodes.parquet')
# edges_file = pathlib.Path('edges.parquet')
nodes_file = input_files_stem / 'nodes.parquet'
edges_file = input_files_stem / 'edges.parquet'

# Processing

For the sake of convenience

In [ ]:
zs_columns = [f'z_{i}' for i in range(1, parameters['model']['dim_z']+1)]
zs_columns[:7]

The *adjacency* matrix

In [ ]:
n_neighbors = 10
# n_neighbors = 5
adj_matrix = sklearn.neighbors.kneighbors_graph(analysis_df[zs_columns].values, n_neighbors=n_neighbors, mode='distance')
adj_matrix.shape

**Similarity** metric (as suggested by JCS)

In [10]:
alpha = 1.
adj_matrix.data = np.exp(- alpha * adj_matrix.data)

A function to build an [igraph](https://igraph.org/) graph from a given similarity matrix

In [11]:
def similarity_matix_to_graph(M: csr_matrix) -> ig.Graph:
    matrix_coo = M.tocoo()
    edges = list(zip(matrix_coo.row, matrix_coo.col))
    weights = matrix_coo.data

    g = ig.Graph(edges=edges, directed=False)
    g.es['weight'] = weights

    return g

G = similarity_matix_to_graph(adj_matrix)

Leidgen algorithm using [leidenalg](https://leidenalg.readthedocs.io/en/stable/index.html) implementation.

In [ ]:
partition = leidenalg.find_partition(G, leidenalg.ModularityVertexPartition)
partition

Commnities are *extracted* from the resulting partition, and its number computed

In [ ]:
communities = partition.membership
n_communities = max(communities) + 1
n_communities

If `DataFrame`s for nodes and edges are found, they are loaded (otherwise, the corresponding variables are set to `None`)

In [ ]:
nodes_df = pd.read_parquet(nodes_file) if nodes_file.exists() else None
if nodes_df is not None:
    print(f'found "{nodes_file}"')

edges_df = pd.read_parquet(edges_file) if edges_file.exists() else None
if edges_df is not None:
    print(f'found "{edges_file}"')

A `DataFrame` with information on the nodes

In [ ]:
if nodes_df is None:

    # so that, for the same data, the results are consistent
    random.seed(42)

    # the *layout* of the nodes when plotting the graph
    layout = G.layout('auto')

    # colors palette
    palette = Category20[20]
    
    
    nodes_df = pd.DataFrame(
        {
            'community': communities,
            'fill_color': [palette[c % 20]for c in communities],
        },
        index=analysis_df.index
    )

    nodes_df[['x', 'y']] = np.array(layout.coords)
    nodes_df['size'] = 8
    nodes_df['alpha'] = 1.

    unknown_series = analysis_df['main_type'].isna()
    nodes_df.loc[unknown_series, 'marker'] = 'square'
    nodes_df.loc[~unknown_series, 'marker'] = 'diamond'

nodes_df

Another one for the edges

In [ ]:
if edges_df is None:

    edges_df = pd.DataFrame(
        [(edge.source, edge.target) for edge in G.es], columns=['start', 'end']
    )

edges_df

The above `DataFrame`s are saved to disk

In [17]:
nodes_df.to_parquet(nodes_file)
edges_df.to_parquet(edges_file)

A convenience function to frame a certain region with a rectangle.

In [18]:
def mark_rectangle(x_range, y_range, fig, text: str):
    
    x_center = np.mean(x_range)
    y_center = np.mean(y_range)
    width = -np.subtract(*x_range)
    height = -np.subtract(*y_range)

    fig.rect(x=x_center, y=y_center, width=width, height=height, line_color='red', line_width=2, fill_alpha=0)

    label = Label(x=x_range[1] + 0.3, y=y_center, text=text, text_font_size="12pt", text_color='black')
    fig.add_layout(label)

    return fig

# Overall view

In [ ]:
fig = trapallada.analysis.plot_graph(analysis_df, nodes_df, edges_df)
show(fig)

A convenience function to select nodes in a certain region

In [20]:
def select_nodes_in_region(x_range, y_range, nodes: pd.DataFrame | None = None):
    
    if nodes is None:
        nodes = nodes_df
    
    select = (x_range[0]< nodes['x']) & (nodes['x'] < x_range[1]) &\
          (y_range[0]< nodes['y']) & (nodes['y'] < y_range[1])
    
    return nodes[select]

Another (relying on the previous one) for plotting a few random points in a region

In [21]:
def sample_and_plot_range(n: int, x_range, y_range, nodes: pd.DataFrame | None = None):

    if nodes is None:
        nodes = nodes_df

    selected = select_nodes_in_region(x_range, y_range, nodes)

    n = min(n, len(selected))

    for i in selected.sample(n).index:
        display(Markdown(f'### {i}'))
        print(analysis_df.filter(regex='^(?!z_|tsne_|set$)').loc[i])
        display(trapallada.analysis.predict_and_plot(model, ds, i, display_metadata=False))

# A few eyeball-selected regions

In [22]:
n_samples = 10

The coordinates (and associated) labels defining the regions

In [23]:
x_range_list = [
    (222, 228),
    (-159, -152),
    (30, 40),
]
y_range_list = [
    (-170, -162),
    (44, 58),
    (-210, -200),
]
labels = [f'Region {i}' for i in range(len(x_range_list))]

On the figure

In [ ]:
for x_range, y_range, label in zip(x_range_list, y_range_list, labels):
    fig = mark_rectangle(x_range, y_range, fig, label)
show(fig)

A loop going through each region

In [ ]:
for x_range, y_range, label in zip(x_range_list, y_range_list, labels):

    display(Markdown(f'### {label}'))
    
    fig.x_range.start, fig.x_range.end = x_range
    fig.y_range.start, fig.y_range.end = y_range
    
    show(fig)

    sample_and_plot_range(n_samples, x_range, y_range)

# Cataclysmic

A convenience to plot all the windows for a given signal (in the above space)

In [26]:
def plot_signal(name: str):
    
    select = (analysis_df['signal'] == name)

    fig = trapallada.analysis.plot_graph(analysis_df, nodes_df, edges_df)

    # fig.renderers[0].node_renderer.level
    
    # for the sake of readability
    data = fig.renderers[0].node_renderer.data_source.data
    
    for i in analysis_df[select].index.tolist():
        data['marker'][i] = 'circle'
        # data['size'][i] = 10
        data['fill_color'][i] = 'magenta'

    for i in analysis_df[~select].index.tolist():
        data['alpha'][i] = 0.0
    
    return fig, analysis_df[select]

In [ ]:
fig, select_df = plot_signal('0840380201/P0840380201PNS003SRCTSR8004')
show(fig)
# display(select_df)

# Javi's paper

All the associated windows

In [ ]:
fig, select_df = plot_signal('0605950101/P0605950101PNS003SRCTSR8001')
show(fig)

A few clusters to zoom in

In [29]:
x_range_list = [
    (-22, 5),
    (115, 140),
    (-195,-180),
    (-100,60),
]
y_range_list = [
    (-10, 5),
    (-170, -155),
    (-180, -170),
    (-240, -190)
]
labels = [f'Cluster {i}' for i in range(len(x_range_list))]

In [ ]:
select = (analysis_df['signal'] == '0605950101/P0605950101PNS003SRCTSR8001')
fig = trapallada.analysis.plot_graph(analysis_df[select], nodes_df[select], edges_df)
for x_range, y_range, label in zip(x_range_list, y_range_list, labels):
    fig = mark_rectangle(x_range, y_range, fig, label)
show(fig)

In [ ]:
fig = trapallada.analysis.plot_graph(analysis_df[select], nodes_df[select], edges_df)

for x_range, y_range, label in zip(x_range_list, y_range_list, labels):

    display(Markdown(f'### {label}'))

    # fig.x_range.start, fig.x_range.end = x_range
    # fig.y_range.start, fig.y_range.end = y_range
    
    # show(fig)
    
    sample_and_plot_range(n_samples, x_range, y_range, nodes_df[select])